# To Catch an AI Liar: Can a linear probe detect deception?

### A  CPU-only tutorial

This notebook is a **small pedagogical adaptation**, not an exact replication, of:

- Goldowsky-Dill et al., *Detecting Strategic Deception with Linear Probes* (ICML 2025);
- Natarajan et al., *One Probe Won't Catch Them All: Towards Targeted Deception Detection* (ICML 2026).

We use pre-collected activations from **Gemma-2-9B-IT**, three on-policy evaluation
sets, and eight instruction-pair probes. No model or GPU is required.


> **Note:**  The first run downloads about 185 MB from
> Hugging Face and may take 1–3 minutes. All helper/plotting code is collapsed; the
> short exercise cells remain visible.




## 0. Setup

The saved files contain activation vectors that were produced earlier on a GPU. In
this notebook we only load those vectors and fit small scikit-learn models on CPU.


In [ ]:
# @title 0A. Install dependencies
# ── Dependencies (safe to run anywhere) ──────────────────────────────────────
# Installs only what is actually missing, so this is a no-op on a machine that
# already has them and a ~15 s setup on a fresh Colab runtime.
import importlib.util
import subprocess
import sys

REQUIRED = ["huggingface_hub", "plotly", "sklearn", "pandas", "numpy"]
PIP_NAME = {"sklearn": "scikit-learn"}

missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    pkgs = [PIP_NAME.get(m, m) for m in missing]
    print("installing:", ", ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("done — if an import fails below, restart the runtime and re-run")
else:
    print("all dependencies already present")


In [ ]:
# @title 0B. Load packages and download the activations
# ── Configuration and imports ────────────────────────────────────────────────
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import (
    StratifiedGroupKFold, StratifiedKFold, cross_val_predict, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
RNG = np.random.default_rng(42)

# ── Where does the data come from? ───────────────────────────────────────────
# Works unchanged on a laptop and on Colab: use a local data/ folder if one is
# anywhere above us, otherwise fetch the published dataset from HuggingFace.
#
# Workshop snapshot tag. For a permanent archival release, replace this tag
# with the immutable commit SHA reported by the Hugging Face dataset page.

HF_REPO_ID = "Rutabin/deception-probing-tutorial-lite"
HF_REVISION = "v1"

DATA_DIR = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "data" / "meta.json").exists()), None)

if DATA_DIR is None:
    from huggingface_hub import snapshot_download
    print(f"No local data/ found -- downloading {HF_REPO_ID}@{HF_REVISION} ...")
    DATA_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset",
                                      revision=HF_REVISION, local_dir="data"))
else:
    print("Using local data (no download needed).")

# A consistent visual language: blue = honest, orange = deceptive.
C_HONEST, C_DECEPTIVE = "#2a78d6", "#eb6834"
C_MUTED, C_GRID, C_INK = "#8a8a85", "#e8e8e4", "#0b0b0b"
SEQ_BLUE = [[0.0, "#cde2fb"], [0.5, "#3987e5"], [1.0, "#0d366b"]]
DIV_AUC = [[0.0, "#e34948"], [0.5, "#f0efec"], [1.0, "#2a78d6"]]

pio.templates["tutorial"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, -apple-system, Segoe UI, sans-serif",
              size=13, color=C_INK),
    paper_bgcolor="#fcfcfb",
    plot_bgcolor="#fcfcfb",
    xaxis=dict(gridcolor=C_GRID, zeroline=False, linecolor=C_GRID),
    yaxis=dict(gridcolor=C_GRID, zeroline=False, linecolor=C_GRID),
    margin=dict(l=70, r=30, t=60, b=55),
    colorway=[C_HONEST, C_DECEPTIVE],
    legend=dict(orientation="h", y=1.06, x=0, bgcolor="rgba(0,0,0,0)"),
))
pio.templates.default = "tutorial"

LABEL_NAME = {0: "honest", 1: "deceptive"}
print("Setup complete. DATA_DIR =", DATA_DIR.resolve())


In [ ]:
# @title 0C. Load vectors and text records
# ── Load the saved vectors and text records ──────────────────────────────────
def load_npz(name):
    path = DATA_DIR / f"{name}.npz"
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as archive:
        data = {key: archive[key] for key in archive.files}
    data["acts"] = data["acts"].astype(np.float32)
    if "acts_last" in data:
        data["acts_last"] = data["acts_last"].astype(np.float32)
    return data


def load_jsonl(name):
    path = DATA_DIR / f"{name}.jsonl"
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


meta_path = DATA_DIR / "meta.json"
if not meta_path.exists():
    raise FileNotFoundError(
        f"Could not find {meta_path}. Set DATA_DIR above, then run this cell again."
    )

META = json.loads(meta_path.read_text())
LAYERS = list(META["layer_subset"])
PRIMARY_LAYER = 20
LI = LAYERS.index(PRIMARY_LAYER)

TRAIN_SETS = ["instructed_pairs"] + [
    f"lietype_{kind}" for kind in
    ["overt_lie", "concealment", "evasion", "exaggeration",
     "partial_truth", "pressure_dishonesty", "white_lie"]
]
EVAL_SETS = ["apollo_sandbagging", "apollo_roleplaying", "apollo_ai_liar"]
CONTROL = "alpaca_control"

D = {name: load_npz(name) for name in TRAIN_SETS + EVAL_SETS + [CONTROL]}
D = {name: data for name, data in D.items() if data is not None}
TEXT = {
    name: load_jsonl(name)
    for name in EVAL_SETS + [CONTROL, "instructed_pairs"]
}

rows = []
for name, data in D.items():
    labels = data["labels"]
    rows.append(dict(
        dataset=name,
        n=len(labels),
        honest=int((labels == 0).sum()),
        deceptive=int((labels == 1).sum()),
        role=("train" if name in TRAIN_SETS else
              "control" if name == CONTROL else "eval"),
        d_model=data["acts"].shape[-1],
    ))

inventory = (
    pd.DataFrame(rows)
    .sort_values(["role", "dataset"])
    .reset_index(drop=True)
)

print(f"Model       : {META['model_name']}")
print(f"Layers kept : {LAYERS}")
print(f"Probe layer : {PRIMARY_LAYER} (stored index {LI})")
print(f"Pooling     : {META['token_rule']}")
print(f"Vector width: {D['instructed_pairs']['acts'].shape[-1]}\n")
inventory


In [ ]:
# @title 0D. Build matched-pair group IDs
# Construct group IDs from the response text. In the instruction-pair dataset,
# the same factual response occurs once under each system instruction. Keeping
# both copies in the same fold prevents matched-pair leakage.
records = TEXT["instructed_pairs"]
record_labels = np.asarray([r["label"] for r in records])
if len(records) != len(D["instructed_pairs"]["labels"]):
    raise ValueError("Text rows and activation rows do not have the same length.")
if not np.array_equal(record_labels, D["instructed_pairs"]["labels"]):
    raise ValueError("Text and activation labels are not in the same row order.")

normalise_response = lambda s: " ".join(str(s).split())
PAIR_GROUPS, pair_texts = pd.factorize(
    np.asarray([normalise_response(r["response"]) for r in records], dtype=object), sort=False
)

pair_check = pd.DataFrame({"group": PAIR_GROUPS, "label": record_labels})
group_sizes = pair_check.groupby("group").size()
group_label_counts = pair_check.groupby("group")["label"].nunique()
complete_pairs = int(((group_sizes == 2) & (group_label_counts == 2)).sum())
print(f"Matched response groups: {len(group_sizes)}; complete honest/deceptive pairs: {complete_pairs}")
if complete_pairs != len(group_sizes):
    print("Warning: some response groups are not complete pairs; they are still kept within folds.")


### Why matched-pair group IDs?

Each response text appears twice in this dataset: once under an honest instruction,
once under a deceptive one. If the two copies land in different
cross-validation folds, the probe can score well by recognising the text instead of
detecting deception. Giving both copies the same group ID lets
`StratifiedGroupKFold` keep them in one fold, which prevents that leak.

## 1. What counts as deception here?

We use **strategic deception** to mean an attempt to induce a false belief in pursuit
of a goal. This differs from:

| Phenomenon | False output possible? | Goal-directed deception required? |
|---|---:|---:|
| Hallucination or mistake | Yes | No |
| Benign role-play or fiction | Yes | No |
| Strategic concealment, misrepresentation, or sandbagging | Yes | Yes, under the operational definition |

The labels below are operational measurements: some are generated by instructions,
some by a rule, and some are assigned by an LLM judge based on the operational definition. They are not direct observations of a
models internal state.


## 2. Where does a probe read the model?

A transformer produces a hidden-state vector for every token position at every
stored layer. The next-token probabilities below are **illustrative**, not outputs
from the model used in the experiment. Use the buttons to see how the available
context changes by position.


In [ ]:
# @title 2A. Explore token positions and the pre-response state
# ── Pick a position: what does the model think comes next? ────────────────────
# Conceptual illustration. No language model is loaded and the percentages are
# illustrative -- the point is *where* hidden states live, not their exact values.
#
# Self-contained: any cell a reader might run first should stand on its own feet.
import textwrap

import plotly.graph_objects as go

# One prompt, already tokenised, plus the two tokens the model went on to generate.
TOKENS = ["<user>", "The", "capital", "of", "France", "is", "<model>", "Paris", "."]
N_PROMPT = 7                 # indices 0-6 are prompt; 7-8 were generated
PRE_RESPONSE = 6             # last prompt token -- the pre-response position

# The top-5 next-token distribution at each position, and what it has to teach.
STATE = [
    dict(nxt=["The", "I", "What", "Can", "How"], p=[.06, .05, .04, .03, .03],
         note="One token of context, and the model has almost no idea what comes "
              "next. The distribution is nearly flat — the bars barely leave the axis."),
    dict(nxt=["first", "best", "same", "most", "capital"], p=[.03, .02, .02, .02, .02],
         note="Still flat. “The” can begin almost any sentence in the language."),
    dict(nxt=["of", "city", "is", "letter", "and"], p=[.78, .07, .03, .02, .01],
         note="Suddenly sharp — but this is grammar, not knowledge. “The capital” is "
              "followed by “of” most of the time, whatever the sentence is about."),
    dict(nxt=["the", "France", "Italy", "a", "Japan"], p=[.31, .05, .03, .03, .02],
         note="A country has to come next, but which one? That needs knowledge rather "
              "than grammar, and the model is genuinely unsure."),
    dict(nxt=["is", "?", ",", "and", "was"], p=[.71, .06, .04, .03, .02],
         note="Back to grammar: “The capital of France” is asking for a verb."),
    dict(nxt=["Paris", "the", "a", "located", "one"], p=[.41, .09, .06, .04, .03],
         note="The interesting one. The answer is already forming inside the prompt, "
              "several tokens before the model is invited to say anything."),
    dict(nxt=["Paris", "The", "It", "France", "Par"], p=[.74, .06, .03, .02, .01],
         note="This is the PRE-RESPONSE position. Not one answer token exists yet, and "
              "the hidden state sitting here already carries the answer. A probe placed "
              "here would read the model before it has spoken."),
    dict(nxt=[".", ",", "is", "—", "!"], p=[.61, .09, .05, .03, .02],
         note="The generated token was appended and fed back in. Every position keeps "
              "its own hidden state; none of the earlier ones are thrown away."),
    dict(nxt=["<end>", "It", "⏎", "The", "Paris"], p=[.88, .03, .02, .01, .01],
         note="The model is ready to stop, and generation ends here. Nine positions, "
              "nine hidden states, any of which we could have photographed."),
]

INK, MUTED, FAINT = "#0b0b0b", "#52514e", "#bdbcb7"
C_PROMPT, C_GEN, C_EDGE = "#cde2fb", "#fbdccd", "#8a8a85"
C_READ = "#0f8a6f"           # the model's own next-token readout (decoding)
PAPER, PANEL, RULE = "#fcfcfb", "#f4f3f0", "#e0dfda"

Y_TOK, Y_BRACKET, Y_NOTE = 0.30, 1.05, -1.08
X0, X1 = -0.75, 8.75

esc = lambda t: t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
CHIP = [esc(t) for t in TOKENS]
# Plotly does not wrap text traces, so wrap them ourselves at a fixed width.
wrap = lambda t: "<br>".join(textwrap.wrap(t, 74))


def variant(sel):
    """The traces that describe 'position `sel` is the one we are looking at'."""
    seen, future = list(range(sel + 1)), list(range(sel + 1, len(TOKENS)))
    s = STATE[sel]
    # Left-anchored, because a centred label overflows the strip when only one
    # token is in context.
    bracket_x, bracket_t, bracket_pos = [X0 + 0.25], ["in context"], ["middle right"]
    if future:
        bracket_x.append(X1 - 0.25)
        bracket_t.append("not yet in the context")
        bracket_pos.append("middle left")
    return [
        # 1. the tokens this position can actually see
        go.Scatter(
            x=seen, y=[Y_TOK] * len(seen), mode="markers+text",
            marker=dict(size=42, symbol="square",
                        color=[C_PROMPT if i < N_PROMPT else C_GEN for i in seen],
                        line=dict(color=C_EDGE, width=1)),
            text=[CHIP[i] for i in seen], textposition="middle center",
            textfont=dict(size=11, color=INK),
            hovertemplate="position %{x} · in context<extra></extra>", showlegend=False),
        # 2. tokens that do not exist yet from this position's point of view
        go.Scatter(
            x=future, y=[Y_TOK] * len(future), mode="markers+text",
            marker=dict(size=42, symbol="square-open", color="#cbcac5",
                        line=dict(width=1.5)),
            text=[f"<span style='color:{FAINT}'>{CHIP[i]}</span>" for i in future],
            textposition="middle center", textfont=dict(size=11),
            hovertemplate="position %{x} · not in the context yet<extra></extra>",
            showlegend=False),
        # 3. the ring marking the position we are reading
        go.Scatter(
            x=[sel], y=[Y_TOK], mode="markers",
            marker=dict(size=54, symbol="square", color="rgba(0,0,0,0)",
                        line=dict(color=C_READ, width=3)),
            hovertemplate="hidden state at position %{x}<extra></extra>",
            showlegend=False),
        # 4. what is / is not in context, said in words above the strip
        go.Scatter(
            x=bracket_x, y=[Y_BRACKET] * len(bracket_x), mode="text", text=bracket_t,
            textfont=dict(size=11, color=MUTED), textposition=bracket_pos,
            hoverinfo="skip", showlegend=False),
        # 5. the per-position caption. A TRACE, not a layout annotation: plotly
        #    merges annotations across updates and the captions end up stacked.
        go.Scatter(
            x=[X0 + 0.25], y=[Y_NOTE], mode="text", text=[wrap(s["note"])],
            textfont=dict(size=12.5, color=MUTED), textposition="middle right",
            hoverinfo="skip", showlegend=False),
        # 6. header over the probability panel, naming the selected token
        go.Scatter(
            x=[0], y=[-1.35], mode="text", xaxis="x2", yaxis="y2",
            text=[f"the token after <b>“{CHIP[sel]}”</b> is…"],
            textfont=dict(size=12.5, color=INK), textposition="middle right",
            hoverinfo="skip", showlegend=False),
        # 7. the distribution itself
        go.Bar(
            x=s["p"], y=list(range(5)), orientation="h",
            marker=dict(color=C_READ, line=dict(width=0)),
            text=[f"  {t} · {p:.0%}" for t, p in zip(map(esc, s["nxt"]), s["p"])],
            textposition="outside", textfont=dict(size=12, color=INK), cliponaxis=False,
            hovertemplate="%{text}<extra></extra>",
            xaxis="x2", yaxis="y2", showlegend=False),
    ]


N_PER = 7
data = [t for sel in range(len(TOKENS)) for t in variant(sel)]

# Always-on: the position numbers, so "position" stays a concrete thing.
data.append(go.Scatter(
    x=list(range(len(TOKENS))), y=[Y_TOK - 0.44] * len(TOKENS), mode="text",
    text=[str(i) for i in range(len(TOKENS))],
    textfont=dict(size=10, color=FAINT), textposition="middle center",
    hoverinfo="skip", showlegend=False))
# Identity is never carried by colour alone, so these are real legend entries.
data += [
    go.Scatter(x=[None], y=[None], mode="markers", name="prompt token",
               marker=dict(size=12, symbol="square", color=C_PROMPT,
                           line=dict(color=C_EDGE, width=1))),
    go.Scatter(x=[None], y=[None], mode="markers", name="token the model generated",
               marker=dict(size=12, symbol="square", color=C_GEN,
                           line=dict(color=C_EDGE, width=1))),
    go.Scatter(x=[None], y=[None], mode="markers", name="not in the context yet",
               marker=dict(size=12, symbol="square-open", color="#cbcac5",
                           line=dict(width=1.5))),
    go.Scatter(x=[None], y=[None], mode="markers",
               name="the position being read",
               marker=dict(size=12, symbol="square", color="rgba(0,0,0,0)",
                           line=dict(color=C_READ, width=2.5))),
]
N_STATIC = 5

TITLE = "Every position has its own hidden state — and every hidden state predicts a next token"
buttons = [
    dict(label=f"  {CHIP[sel]}  ", method="update",
         args=[{"visible": [i // N_PER == sel for i in range(N_PER * len(TOKENS))]
                           + [True] * N_STATIC},
               {"title.text": TITLE + (
                   "  ·  reading the PRE-RESPONSE position" if sel == PRE_RESPONSE
                   else f"  ·  reading position {sel}")}])
    for sel in range(len(TOKENS))
]

fig = go.Figure(data=data)
for trace, on in zip(fig.data, [i // N_PER == PRE_RESPONSE
                                for i in range(N_PER * len(TOKENS))] + [True] * N_STATIC):
    trace.visible = on

fig.update_layout(
    title=dict(text=TITLE + "  ·  reading the PRE-RESPONSE position",
               x=0.005, y=0.975, yanchor="top", font=dict(size=14.5)),
    height=452, margin=dict(l=22, r=22, t=132, b=74),
    paper_bgcolor=PAPER, plot_bgcolor=PAPER, dragmode=False,
    font=dict(family="Inter, -apple-system, Segoe UI, sans-serif", color=INK),
    legend=dict(orientation="h", y=-0.10, x=0, bgcolor="rgba(0,0,0,0)",
                font=dict(size=11)),
    xaxis=dict(domain=[0.0, 0.615], range=[X0, X1], visible=False, anchor="y"),
    yaxis=dict(domain=[0.0, 1.0], range=[-1.80, 1.42], visible=False, anchor="x"),
    xaxis2=dict(domain=[0.685, 1.0], range=[0, 1.36], tickvals=[0, 0.5, 1.0],
                tickformat=".0%", gridcolor="#e8e8e4", anchor="y2",
                ticks="outside", tickcolor=RULE, tickfont=dict(size=10.5, color=MUTED)),
    yaxis2=dict(domain=[0.16, 0.84], range=[4.6, -1.7], visible=False, anchor="x2"),
    # A panel behind the caption keeps it visibly separate from the token strip.
    shapes=[dict(type="rect", xref="x", yref="y", layer="below",
                 x0=X0, x1=X1, y0=-1.72, y1=-0.52,
                 fillcolor=PANEL, line=dict(color=RULE, width=1))],
    updatemenus=[dict(
        type="buttons", direction="right", active=PRE_RESPONSE,
        x=-0.004, y=1.10, xanchor="left", yanchor="bottom", pad=dict(r=0, t=0),
        bgcolor=PAPER, bordercolor="#d8d7d2", borderwidth=1,
        font=dict(size=11.5, color=INK), showactive=True, buttons=buttons)],
    annotations=[dict(x=0, y=1.30, xref="paper", yref="paper", xanchor="left",
                      yanchor="bottom", showarrow=False,
                      font=dict(size=11.5, color=MUTED),
                      text="Pick a token — each button reads the hidden state at that "
                           "position and shows what it predicts:")],
)
fig.show(config=dict(displayModeBar=False, responsive=True))


In [ ]:
# @title 2B. The residual stream and the probe position
# ── Static architecture diagram: the residual stream ─────────────────────────
# Pure SVG built as a string -- no image files, no external assets, so it renders
# in Colab, in nbviewer, and in an exported HTML with nothing to fetch.

# Self-contained: this cell runs BEFORE the shared imports cell, so it brings its
# own.
from pathlib import Path

from IPython.display import SVG, display

W, H = 950, 460
BELT_Y, BELT_H = 250, 34
X_IN, X_OUT = 132, 812
LAYER_X = [200, 290, 380, 470, 560, 650, 740]     # 7 sampled layer positions
LAYER_LABEL = ["0", "6", "13", "20", "27", "34", "41"]
TAP = 3                                            # index of layer 20

INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d7d2"
BELT, BELT_EDGE = "#cde2fb", "#86b6ef"
BLOCK, TAPC, ACCENT = "#f0efec", "#4a3aa7", "#2a78d6"

parts = [
    f'<rect width="{W}" height="{H}" fill="#fcfcfb"/>',
    f'<text x="24" y="34" font-size="16" font-weight="700" fill="{INK}">'
    f'The residual stream — a conveyor belt running through 42 layers</text>',
    f'<text x="24" y="56" font-size="12.5" fill="{MUTED}">'
    f'Every layer READS from the belt and ADDS an update. Earlier contributions may persist, '
    f'but later layers can transform or cancel them.</text>',
    # the belt
    f'<rect x="{X_IN}" y="{BELT_Y}" width="{X_OUT - X_IN}" height="{BELT_H}" rx="6" '
    f'fill="{BELT}" stroke="{BELT_EDGE}" stroke-width="1.5"/>',
    # Caption goes in the footer, NOT under the belt: the read/write arrows live
    # there and text on top of them was unreadable.
    f'<text x="{W/2:.0f}" y="{H - 20}" font-size="12.5" text-anchor="middle" '
    f'fill="{ACCENT}" font-weight="600">'
    f'the residual stream &#183; one vector of 3,584 numbers per token, per layer</text>',
    # input
    f'<text x="{X_IN - 12}" y="{BELT_Y + 14}" font-size="12" text-anchor="end" fill="{INK}">token</text>',
    f'<text x="{X_IN - 12}" y="{BELT_Y + 29}" font-size="12" text-anchor="end" fill="{INK}">embedding</text>',
    # output
    f'<rect x="{X_OUT + 14}" y="{BELT_Y - 12}" width="98" height="58" rx="6" '
    f'fill="{BLOCK}" stroke="{RULE}"/>',
    f'<text x="{X_OUT + 63}" y="{BELT_Y + 10}" font-size="12" text-anchor="middle" '
    f'fill="{INK}" font-weight="600">output head</text>',
    f'<text x="{X_OUT + 63}" y="{BELT_Y + 28}" font-size="11" text-anchor="middle" '
    f'fill="{MUTED}">next token</text>',
    f'<path d="M {X_OUT} {BELT_Y + BELT_H/2} L {X_OUT + 10} {BELT_Y + BELT_H/2}" '
    f'stroke="{MUTED}" stroke-width="2" marker-end="url(#ar)"/>',
    '<defs><marker id="ar" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">'
    f'<path d="M0,0 L6,3 L0,6 Z" fill="{MUTED}"/></marker>'
    '<marker id="art" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">'
    f'<path d="M0,0 L6,3 L0,6 Z" fill="{TAPC}"/></marker></defs>',
]

for i, x in enumerate(LAYER_X):
    tapped = (i == TAP)
    col = TAPC if tapped else RULE
    wgt = "2" if tapped else "1"
    # attention block above, MLP block below -- both read from and write to the belt
    parts += [
        f'<rect x="{x-34}" y="{BELT_Y-92}" width="68" height="46" rx="5" fill="{BLOCK}" '
        f'stroke="{col}" stroke-width="{wgt}"/>',
        f'<text x="{x}" y="{BELT_Y-64}" font-size="10.5" text-anchor="middle" fill="{INK}">attention</text>',
        f'<rect x="{x-34}" y="{BELT_Y+BELT_H+46}" width="68" height="46" rx="5" fill="{BLOCK}" '
        f'stroke="{col}" stroke-width="{wgt}"/>',
        f'<text x="{x}" y="{BELT_Y+BELT_H+74}" font-size="10.5" text-anchor="middle" fill="{INK}">MLP</text>',
        # read (belt -> block) and write (block -> belt)
        f'<path d="M {x-12} {BELT_Y} L {x-12} {BELT_Y-46}" stroke="{MUTED}" stroke-width="1.4" '
        f'marker-end="url(#ar)" fill="none"/>',
        f'<path d="M {x+12} {BELT_Y-46} L {x+12} {BELT_Y}" stroke="{MUTED}" stroke-width="1.4" '
        f'marker-end="url(#ar)" fill="none"/>',
        f'<path d="M {x-12} {BELT_Y+BELT_H} L {x-12} {BELT_Y+BELT_H+46}" stroke="{MUTED}" '
        f'stroke-width="1.4" marker-end="url(#ar)" fill="none"/>',
        f'<path d="M {x+12} {BELT_Y+BELT_H+46} L {x+12} {BELT_Y+BELT_H}" stroke="{MUTED}" '
        f'stroke-width="1.4" marker-end="url(#ar)" fill="none"/>',
        f'<text x="{x}" y="{BELT_Y+BELT_H+112}" font-size="10.5" text-anchor="middle" '
        f'fill="{TAPC if tapped else MUTED}" font-weight="{"700" if tapped else "400"}">'
        f'layer {LAYER_LABEL[i]}</text>',
    ]

# ellipses between sampled layers
for x in [(LAYER_X[i] + LAYER_X[i+1]) / 2 for i in range(len(LAYER_X) - 1)]:
    parts.append(f'<text x="{x:.0f}" y="{BELT_Y+BELT_H+112}" font-size="11" '
                 f'text-anchor="middle" fill="{RULE}">&#183;&#183;&#183;</text>')

# the tap callout
tx = LAYER_X[TAP]
gap_x = (LAYER_X[TAP] + LAYER_X[TAP + 1]) / 2      # clear corridor between blocks
cy = BELT_Y + BELT_H / 2
box_y = 96
parts += [
    f'<circle cx="{tx}" cy="{cy}" r="13" fill="none" stroke="{TAPC}" stroke-width="2.5"/>',
    # L-shaped leader: out of the ring, along the belt, then up through the gap.
    # A straight vertical line would have crossed the attention block and read as
    # "we photograph the attention output", which is not what a probe does.
    f'<path d="M {tx+13} {cy} L {gap_x:.0f} {cy} L {gap_x:.0f} {box_y + 62}" '
    f'stroke="{TAPC}" stroke-width="2" fill="none" marker-end="url(#art)" '
    f'stroke-dasharray="5 3"/>',
    f'<rect x="{gap_x-165:.0f}" y="{box_y}" width="330" height="58" rx="6" fill="#fcfcfb" '
    f'stroke="{TAPC}" stroke-width="1.5"/>',
    f'<text x="{gap_x:.0f}" y="{box_y+24}" font-size="12.5" text-anchor="middle" fill="{TAPC}" '
    f'font-weight="700">we photograph the belt here &#8212; layer 20</text>',
    f'<text x="{gap_x:.0f}" y="{box_y+43}" font-size="11.5" text-anchor="middle" fill="{MUTED}">'
    f'3,584 numbers &#8594; logistic regression &#8594; honest / deceptive</text>',
]

svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="{H}" ' \
      f'viewBox="0 0 {W} {H}" role="img" aria-label="Diagram of a transformer residual ' \
      f'stream with a probe tapping layer 20">' + "".join(parts) + '</svg>'

display(SVG(svg))


### Layer, position, and pooling are separate choices

This tutorial follows Natarajan et al.: use stored layer 20, average response-token
states, and exclude the final five response tokens. Excluding the ending reduces
direct leakage from the resolved fact. It is a design intended to focus the probe on
instruction-conditioned state, not a direct proof that the remaining signal is “intent.”


In [ ]:
# @title 2C. Which response positions are pooled?
# ── Where the probe actually looks ───────────────────────────────────────────
# Illustrative token strip for one training example, drawn in SVG so nothing can
# collide. The three regions correspond exactly to `token_rule` in meta.json.
from pathlib import Path

from IPython.display import SVG, display

# "tpl" = prompt / chat template   "pre" = last template token (the pre-response
# position)   "use" = pooled       "cut" = the excluded final five
TOKENS = [
    ("<start_of_turn>", "tpl"), ("user", "tpl"), ("…", "tpl"), ("<end_of_turn>", "tpl"),
    ("<start_of_turn>", "tpl"), ("model", "pre"),
    ("The", "use"), ("blue", "use"), ("whale", "use"),
    ("is", "cut"), ("the", "cut"), ("largest", "cut"), ("mammal", "cut"), (".", "cut"),
]

FILL = {"tpl": "#f0efec", "pre": "#e6e2f6", "use": "#cde2fb", "cut": "#fcfcfb"}
EDGE = {"tpl": "#d8d7d2", "pre": "#4a3aa7", "use": "#2a78d6", "cut": "#eb6834"}
INK, MUTED = "#0b0b0b", "#52514e"

W, PAD, GAP, H_CHIP, Y_ROW, H = 980, 9, 5, 34, 168, 420
esc = lambda t: t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

# lay the chips out on one row and remember where each landed
pos, x = [], 24
for text, kind in TOKENS:
    w = int(6.8 * len(text)) + 2 * PAD
    pos.append((x, w, text, kind))
    x += w + GAP

parts = [
    f'<rect width="{W}" height="{H}" fill="#fcfcfb"/>',
    f'<text x="24" y="32" font-size="16" font-weight="700" fill="{INK}">'
    f'Which token positions go into the vector?</text>',
    f'<text x="24" y="54" font-size="12.5" fill="{MUTED}">'
    f'The probe averages the blue response tokens. '
    f' </text>',
]


def span(kinds):
    """x-extent of the contiguous run of chips whose kind is in `kinds`."""
    sel = [p for p in pos if p[3] in kinds]
    return sel[0][0], sel[-1][0] + sel[-1][1]


def bracket(x0, x1, y, label, color, sub=None, below=False, dy=0):
    # `dy` staggers labels of adjacent brackets. A narrow bracket's centred label
    # overflows its own span, so two side-by-side labels collide unless one drops.
    mid, tick = (x0 + x1) / 2, (10 if below else -10)
    out = [f'<path d="M {x0} {y - tick} L {x0} {y} L {x1} {y} L {x1} {y - tick}" fill="none" '
           f'stroke="{color}" stroke-width="1.5"/>',
           f'<text x="{mid:.0f}" y="{y + dy + (26 if below else -10)}" font-size="12" '
           f'text-anchor="middle" fill="{color}" font-weight="600">{label}</text>']
    if sub:
        out.append(f'<text x="{mid:.0f}" y="{y + (43 if below else -27)}" font-size="11" '
                   f'text-anchor="middle" fill="{MUTED}">{sub}</text>')
    return out


# brackets above the strip
parts += bracket(*span({"tpl", "pre"}), Y_ROW - 18,
                 "prompt + chat template &#8212; NOT pooled", "#8a8a85",
                 "a &#8220;pre-response&#8221; probe would read the last of these")
# brackets below the strip, so the two groups never collide
y_below = Y_ROW + H_CHIP + 14
parts += bracket(*span({"use"}), y_below,
                 "averaged &#8594; the 3,584-number vector", "#2a78d6", below=True)
parts += bracket(*span({"cut"}), y_below, "last 5 &#8212; EXCLUDED", "#eb6834",
                 below=True, dy=22)

for x0, w, text, kind in pos:
    dash = ' stroke-dasharray="4 3"' if kind == "cut" else ""
    parts += [
        f'<rect x="{x0}" y="{Y_ROW}" width="{w}" height="{H_CHIP}" rx="5" '
        f'fill="{FILL[kind]}" stroke="{EDGE[kind]}" stroke-width="1.5"{dash}/>',
        f'<text x="{x0 + w / 2:.0f}" y="{Y_ROW + 22}" font-size="11.5" text-anchor="middle" '
        f'fill="{INK}">{esc(text)}</text>',
    ]
    if kind == "cut":
        parts.append(f'<line x1="{x0 + 6}" y1="{Y_ROW + H_CHIP / 2}" x2="{x0 + w - 6}" '
                     f'y2="{Y_ROW + H_CHIP / 2}" stroke="{EDGE["cut"]}" stroke-width="1.2"/>')

yl = y_below + 100
parts += [
    f'<text x="24" y="{yl}" font-size="12.5" fill="{INK}" font-weight="600">'
    f'token_rule = mean_response_excl_last_5</text>',
    f'<text x="24" y="{yl + 24}" font-size="12" fill="{MUTED}">'
    f'The last five are cut so the claim has not yet resolved. At &#8220;The blue whale '
    f'is the&#8230;&#8221; the model has not committed to</text>',
    f'<text x="24" y="{yl + 42}" font-size="12" fill="{MUTED}">'
    f'&#8220;largest&#8221; or &#8220;smallest&#8221;, which reduces direct truth-value leakage '
    f'from the finished sentence. It does not by itself identify intent.</text>',
]

svg = (f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="{H}" '
       f'viewBox="0 0 {W} {H}" role="img" aria-label="Token strip showing which positions '
       f'the probe averages">' + "".join(parts) + "</svg>")

display(SVG(svg))


## 3. How data is generated?

The experimental distinction is off policy vs. on policy generation:

- **Off-policy/token-forced training:** forward pass of the deceptive or honest response. Label is assigned based on which system instruction was present. The model did not choose the response.
- **On-policy evaluation:** Gemma samples a response at temperature 1, and a rule or grader labels it.

In [ ]:
# @title 3B. Token-forced training versus on-policy evaluation
# ── Two ways a labeled example gets made ─────────────────────────────────────
# The single most important structural fact in this notebook, and the easiest to
# lose: the probe is TRAINED on responses the model never chose to produce, and
# TESTED on responses it did. Drawn as SVG so nothing can collide or reflow.
from pathlib import Path

from IPython.display import SVG, display

W, H = 1000, 486
INK, MUTED = "#0b0b0b", "#52514e"
BLUE, ORANGE, PAPER = "#2a78d6", "#eb6834", "#fcfcfb"

LANES = [
    dict(y=104, accent="#4a3aa7", tint="#eeecf9",
         tag="TRAIN  ·  off-policy, token-forced",
         boxes=[("two system prompts", "“be honest” / “be deceptive”"),
                ("one response, pasted in", "IDENTICAL in both members"),
                ("Gemma reads it", "we save the activations"),
                ("label = which instruction", "known by construction · no judge")],
         note=("The model never chose anything. The response is fixed, so there is no "
               "behavioural evaluation."),
         note2="Important for producing balanced labeled data set for training."),
    dict(y=292, accent=ORANGE, tint="#fdeee7",
         tag="TEST  ·  on-policy, free generation",
         boxes=[("a scenario prompt", "from deception benchmarks"),
                ("Gemma generates", "sampled at temperature 1"),
                ("a grader reads it", "LLM judge, or a rule"),
                ("label = what it did", "ambiguous verdicts dropped")],
         note=("Here the model chose. The responses labeled deceptive is a behavioral feature."),
         note2=""),
]

X0, X1, GAP, BH = 26, W - 26, 16, 74
esc = lambda t: t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
parts = [f'<rect width="{W}" height="{H}" fill="{PAPER}"/>',
         f'<text x="{X0}" y="34" font-size="17" font-weight="700" fill="{INK}">'
         f'Two ways a labeled deception example gets made</text>',
         f'<text x="{X0}" y="58" font-size="12.5" fill="{MUTED}">'
         f'Same model, same pooling rule, same layer. What differs is who wrote the '
         f'response and therefore what the label can mean.</text>']

for lane in LANES:
    y, acc = lane["y"], lane["accent"]
    parts += [
        f'<rect x="{X0}" y="{y - 34}" width="{int(7.0 * len(lane["tag"])) + 22}" height="23" '
        f'rx="11" fill="{lane["tint"]}" stroke="{acc}" stroke-width="1.2"/>',
        f'<text x="{X0 + 11}" y="{y - 18}" font-size="12" font-weight="700" fill="{acc}">'
        f'{esc(lane["tag"])}</text>',
    ]
    n = len(lane["boxes"])
    bw = (X1 - X0 - GAP * (n - 1)) / n
    for i, (head, sub) in enumerate(lane["boxes"]):
        bx = X0 + i * (bw + GAP)
        # The last box is the payoff -- the label -- so it gets the filled style.
        last = i == n - 1
        parts += [
            f'<rect x="{bx:.0f}" y="{y}" width="{bw:.0f}" height="{BH}" rx="7" '
            f'fill="{lane["tint"] if last else PAPER}" stroke="{acc}" '
            f'stroke-width="{2 if last else 1.3}"/>',
            f'<text x="{bx + bw / 2:.0f}" y="{y + 30}" font-size="13" font-weight="600" '
            f'text-anchor="middle" fill="{INK}">{esc(head)}</text>',
            f'<text x="{bx + bw / 2:.0f}" y="{y + 51}" font-size="11.5" '
            f'text-anchor="middle" fill="{MUTED}">{esc(sub)}</text>',
        ]
        if i:  # arrow in the gap: from the previous box's right edge to this one
            x_from, x_to = bx - GAP + 3, bx - 4
            cy = y + BH / 2
            parts.append(f'<path d="M {x_from:.0f} {cy} L {x_to:.0f} {cy} '
                         f'M {x_to - 5:.0f} {cy - 4} L {x_to:.0f} {cy} L {x_to - 5:.0f} {cy + 4}" '
                         f'fill="none" stroke="{acc}" stroke-width="1.6" '
                         f'stroke-linecap="round" stroke-linejoin="round"/>')
    parts += [
        f'<text x="{X0}" y="{y + BH + 26}" font-size="12" fill="{INK}">{esc(lane["note"])}</text>',
        f'<text x="{X0}" y="{y + BH + 44}" font-size="12" fill="{INK}">{esc(lane["note2"])}</text>',
    ]

parts += [
    f'<line x1="{X0}" y1="{H - 62}" x2="{X1}" y2="{H - 62}" stroke="#e8e8e4" stroke-width="1"/>',
    f'<text x="{X0}" y="{H - 38}" font-size="12.5" font-weight="600" fill="{INK}">'
    f''
    f'</text>',
    f'<text x="{X0}" y="{H - 18}" font-size="12.5" fill="{MUTED}">'
    f' '
    f'</text>',
]

svg = (f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="{H}" '
       f'viewBox="0 0 {W} {H}" role="img" aria-label="Off-policy token-forced training '
       f'versus on-policy evaluation">' + "".join(parts) + "</svg>")
display(SVG(svg))


In [ ]:
# @title 3A. Inspect one pair, one on-policy example, and one control (explore data in depth in Appendix A.)
def clip(text, n=240):
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + " …"


# One matched token-forced pair.
pair_group = next(
    g for g in np.unique(PAIR_GROUPS)
    if set(record_labels[PAIR_GROUPS == g]) == {0, 1}
)
pair_rows = [records[i] for i in np.flatnonzero(PAIR_GROUPS == pair_group)]
print("MATCHED TRAINING PAIR — identical pasted response")
for r in sorted(pair_rows, key=lambda x: x["label"]):
    print(f"\n{LABEL_NAME[r['label']].upper()} instruction: {clip(r['system'])}")
    print("response:", clip(r["response"]))


# One compact on-policy example and one ordinary control response.
for name in ["apollo_roleplaying", "alpaca_control"]:
    r = TEXT[name][0]
    print(f"\n{name.upper()}")
    print("system:", clip(r.get("system", "")))
    print("user:  ", clip(r.get("user", "")))
    print("model: ", clip(r.get("response", "")))
    print("label: ", LABEL_NAME.get(r.get("label"), r.get("label")))


In [ ]:
# @title 3C. Candidate confounds in the datasets
# ── How long are the responses, and how varied are the prompts? ──────────────
# A compact structural summary. Differences here are candidate confounds.

rows = []
for name in ["instructed_pairs", "apollo_sandbagging", "apollo_roleplaying",
             "apollo_ai_liar", "alpaca_control"]:
    records = TEXT.get(name, [])
    if not records:
        continue
    lengths = np.array([len(str(r["response"]).split()) for r in records])
    systems = {r["label"]: set() for r in records}
    for r in records:
        systems[r["label"]].add(r["system"])
    shared = (set.intersection(*systems.values())
              if len(systems) > 1 else set())
    rows.append(dict(
        dataset=name,
        n=len(records),
        median_response_words=int(np.median(lengths)),
        p90_response_words=int(np.percentile(lengths, 90)),
        distinct_system_prompts=len(set(r["system"] for r in records)),
        prompts_shared_across_labels=len(shared),
    ))

structure = pd.DataFrame(rows)
display(structure)

print("""
 A value of 0 at `prompts_shared_across_labels` means honest and
deceptive examples never share a system prompt so a classifier could in principle
succeed by recognising the prompt alone, without representing deception at all.
Only sandbagging has a non-zero value here. This makes exact prompt identity a
less available shortcut there, but it does not prove that prompt variation causes
the performance difference; response length, task and label source also differ.
""")


## 4. Train and understand a linear probe

Logistic regression learns a score \(s(x)=w^\top z+b\) in standardized activation
space. The direction \(w\) is supervised: it is chosen to separate the labels.

**AUC** (area under the ROC curve) is the probability that a randomly selected deceptive
example receives a higher score than a randomly selected honest example. AUC 0.5 is
chance. AUC below 0.5 is a systematically reversed ranking.


### Exercise 1 — What does a high training  AUC establish?

Suppose grouped cross-validation on the token-forced training data gives AUC ≈ 1.00. Which conclusions follow?

1. The training prompt condition is linearly decodable from these activations.
2. Gemma would have generated a lie in the deceptive condition.
3. The probe will generalize to sandbagging and role-playing.
4. The direction is a causal mechanism that the model uses to deceive.

<details>
<summary><b>Reveal answer and explanations</b></summary>

**Only statement 1 follows directly.**

- **1 is supported:** held-out examples can be ranked by the training label using a
  linear function of their activations.
- **2 is unsupported:** the response was pasted in. The model made no behavioral
  choice, so generation propensity was never measured.
- **3 is unsupported:** cross-validation is within the same construction. Transfer
  must be tested on separately generated on-policy data.
- **4 is unsupported:** a probe establishes linear decodability, not that the model
  uses the decoded direction causally. That would require interventions or other
  causal evidence.

</details>


In [ ]:
# @title 4A. Probe helpers
def make_probe(C=1.0):
    return Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(C=C, max_iter=1000, random_state=42)),
    ])


def fit_probe(dataset, layer_index=LI, key="acts", C=1.0):
    data = D[dataset]
    probe = make_probe(C)
    probe.fit(data[key][:, layer_index, :], data["labels"])
    return probe


def probe_scores(probe, dataset, layer_index=LI, key="acts", probabilities=True):
    X = D[dataset][key][:, layer_index, :]
    if probabilities:
        return probe.predict_proba(X)[:, 1]
    return probe.decision_function(X)


def cv_auc(dataset, layer_index=LI, key="acts", folds=5, labels=None):
    data = D[dataset]
    X = data[key][:, layer_index, :]
    y = data["labels"] if labels is None else np.asarray(labels)
    if dataset == "instructed_pairs":
        cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=42)
        return cross_val_score(
            make_probe(), X, y, groups=PAIR_GROUPS, cv=cv, scoring="roc_auc"
        )
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    return cross_val_score(make_probe(), X, y, cv=cv, scoring="roc_auc")


Four helpers used by every experiment below: `make_probe` defines the probe
(standardise, then logistic regression), `fit_probe` trains one on a chosen dataset
and layer, `probe_scores` applies a frozen probe to new data, and `cv_auc` estimates held-out AUC, using group-aware folds for
the matched-pair training set and plain stratified folds elsewhere.

In [ ]:
scores = cv_auc("instructed_pairs")
baseline_probe = fit_probe("instructed_pairs")

print(f"Instruction-pair probe, stored layer {PRIMARY_LAYER}")
print(f"Grouped 5-fold CV AUC : {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Training examples     : {len(D['instructed_pairs']['labels'])}")
print(f"Matched groups        : {len(np.unique(PAIR_GROUPS))}")


### Exercise 2 — Label-shuffle sanity check

If the labels carry the real association, destroying that association should return
cross-validated AUC to approximately 0.5. To test this assumption we assign random labels and train the probe. Predict the result, then run the cell.


In [ ]:
# Try a different seed after the first run. A single finite sample will not land
# on exactly 0.500, but it should be close rather than near the real-label score.
SHUFFLE_SEED = 7
rng_shuffle = np.random.default_rng(SHUFFLE_SEED)
shuffled_labels = rng_shuffle.permutation(D["instructed_pairs"]["labels"])
shuffle_scores = cv_auc("instructed_pairs", labels=shuffled_labels)

print(f"Real labels     : {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Shuffled labels : {shuffle_scores.mean():.3f} ± {shuffle_scores.std():.3f}")


## 4 B. What the probe actually computes

Standardize the activation vector \(x\), using statistics from the training fold, then
reduce the resulting 3,584-dimensional vector to one number:

$
z_j=\frac{x_j-\mu_j}{\sigma_j},
\qquad
s(z)=w^\top z+b,
\qquad
P(y=1\mid z)=\operatorname{sigmoid}(s(z)).
$

The dot product $(w^\top z)$ says where the activation are along one oriented axis.
Everything perpendicular to that axis is discarded.

Splitting \(w\) into direction and length separates three things the probe does:

$
s(z)=\| w\|\left(u^\top z-t\right),
\qquad
u=\frac{w}{\lVert w\rVert},
\qquad
t=-\frac{b}{\lVert w\rVert}.
$

- \(u\) is the **direction** used to rank examples;
- \(t\) is the **threshold location** along that direction;
- $\lVert w\rVert$ controls the **scale** of the logistic score.

Only \(u\) affects ROC-AUC: adding \(b\) or multiplying all scores by a positive
constant leaves the ordering unchanged. The threshold matters for an operational
detector, because it determines which responses are flagged.

The figure below is a two-dimensional toy version of the same operation. **Left:**
each point is projected perpendicularly onto the chosen direction. **Upper right:**
those projections become one-dimensional probe-score distributions. **Lower right:**
AUC records how well that direction ranks deceptive examples above honest ones.

Move the direction through 180°. The line is the same, but its orientation is
reversed, so high and low scores exchange meanings. This is why an AUC below 0.5 can
represent a systematically reversed signal rather than an absence of information.

Logistic regression fits \(w\) and \(b\) by minimizing regularized log loss. It does
not directly search over angles or maximize AUC. In the toy example, the fitted
direction discounts directions with large within-class variation: separation is
useful only when it is large relative to the noise.

In [ ]:
# @title Setup for the figure 2-D toy data
# Setup for the figure below: a 2-D toy dataset and two candidate directions.
#
# The noise cloud is SHARED by both classes and strongly stretched along a tilted
# axis -- the realistic case, where activations vary far more along some directions
# (topic, length, style) than others. The two classes are then offset along x.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(7)
N = 420

tilt = np.deg2rad(48)
R = np.array([[np.cos(tilt), -np.sin(tilt)], [np.sin(tilt), np.cos(tilt)]])
SPREAD = R @ np.diag([2.9, 0.42])

label = rng.integers(0, 2, N)                       # 0 = honest, 1 = deceptive
X = rng.normal(size=(N, 2)) @ SPREAD.T + (2 * label - 1)[:, None] * np.array([0.78, 0.0])

unit = lambda v: v / np.linalg.norm(v)

# Two candidates the figure marks on the AUC curve.
#   difference of means -- the intuitive guess: point from one class centre to the
#     other. It ignores the shape of the noise.
#   fitted probe        -- what logistic regression finds. It accounts for the
#     shared stretch, so it maximises separation RELATIVE TO the noise.
w_diff = unit(X[label == 1].mean(0) - X[label == 0].mean(0))
w_fit = unit(LogisticRegression(max_iter=5000).fit(X, label).coef_[0])

# AUC for every direction. A direction is just an angle in 2-D, so we can try them
# all -- in 3,584 dimensions we cannot, which is why the probe is FITTED, not searched.
angles = np.arange(0, 360, 2)
auc_by_angle = np.array([
    roc_auc_score(label, X @ np.array([np.cos(np.deg2rad(a)), np.sin(np.deg2rad(a))]))
    for a in angles
])

ang_of = lambda w: np.degrees(np.arctan2(w[1], w[0])) % 360

In [ ]:
# @title Figure — rotate the probe direction { display-mode: "form" }

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score


def direction_figure(n_show=13, step_deg=5):
    """Show scalar projection and AUC for every 2-D unit direction."""

    INK, MUTED, RULE, PAPER = "#0b0b0b", "#52514e", "#e0dfda", "#fcfcfb"
    C_HONEST, C_DECEPTIVE, C_PROBE = "#2a78d6", "#eb6834", "#4a3aa7"

    frame_deg = np.arange(0, 360, step_deg)

    def unit_at(angle):
        radians = np.deg2rad(angle)
        return np.array([np.cos(radians), np.sin(radians)])

    # Keep both panels on fixed scales. Otherwise rotating the direction could
    # make a narrow distribution look as wide as a noisy one.
    xy_radius = max(5.0, 1.08 * np.linalg.norm(X, axis=1).max())
    score_radius = 1.05 * np.linalg.norm(X, axis=1).max()
    score_range = (-score_radius, score_radius)

    edges = np.linspace(*score_range, 33)
    centres = (edges[:-1] + edges[1:]) / 2
    bin_width = edges[1] - edges[0]

    shown = np.linspace(0, len(X) - 1, n_show).astype(int)
    n_honest = max(1, np.sum(label == 0))
    n_deceptive = max(1, np.sum(label == 1))

    def projected_parts(angle):
        u = unit_at(angle)
        projection = X @ u
        auc = roc_auc_score(label, projection)

        # Drop selected points perpendicularly onto the u axis.
        segment_x, segment_y = [], []
        for i in shown:
            foot = projection[i] * u
            segment_x.extend([X[i, 0], foot[0], None])
            segment_y.extend([X[i, 1], foot[1], None])

        # This is an illustrative midpoint threshold, not the fitted intercept.
        threshold = 0.5 * (
            projection[label == 0].mean()
            + projection[label == 1].mean()
        )

        perpendicular = np.array([-u[1], u[0]])
        # Three points, not two: the middle one is only there to carry the
        # "u·x = t" label at a position that stays inside the visible range.
        boundary = np.array([
            threshold * u - 2 * xy_radius * perpendicular,
            threshold * u + 0.62 * xy_radius * perpendicular,
            threshold * u + 2 * xy_radius * perpendicular,
        ])

        # Normalize within each class so unequal class sizes do not affect height.
        honest_hist = (
            np.histogram(projection[label == 0], bins=edges)[0]
            / n_honest
        )
        deceptive_hist = (
            np.histogram(projection[label == 1], bins=edges)[0]
            / n_deceptive
        )

        return (
            u,
            projection,
            auc,
            segment_x,
            segment_y,
            boundary,
            honest_hist,
            deceptive_hist,
        )

    # Fix the histogram y-axis across every possible slider position.
    hist_peak = 0.0
    for angle in frame_deg:
        *_, honest_hist, deceptive_hist = projected_parts(angle)
        hist_peak = max(
            hist_peak,
            honest_hist.max(),
            deceptive_hist.max(),
        )

    hist_ymax = 1.15 * hist_peak
    arrow_length = 0.72 * xy_radius

    def dynamic_traces(angle):
        (
            u,
            projection,
            auc,
            segment_x,
            segment_y,
            boundary,
            honest_hist,
            deceptive_hist,
        ) = projected_parts(angle)

        return [
            # Unit probe direction u.
            go.Scatter(
                x=[0, arrow_length * u[0]],
                y=[0, arrow_length * u[1]],
                mode="lines+markers+text",
                line=dict(color=C_PROBE, width=3.5),
                marker=dict(size=[0, 13], color=C_PROBE),
                text=["", "<b>u</b>"],
                textposition="top center",
                textfont=dict(size=15, color=C_PROBE),
                hovertemplate="unit probe direction u<extra></extra>",
                showlegend=False,
            ),

            # Illustrative threshold u·x = t.
            go.Scatter(
                x=boundary[:, 0],
                y=boundary[:, 1],
                mode="lines+text",
                line=dict(color=INK, width=1.6, dash="dash"),
                text=["", "u·x = t", ""],
                textposition="top right",
                textfont=dict(size=12, color=INK),
                hovertemplate=(
                    "illustrative midpoint threshold<br>"
                    "u·x = t; perpendicular to u<extra></extra>"
                ),
                showlegend=False,
            ),

            # Perpendicular projection segments.
            go.Scatter(
                x=segment_x,
                y=segment_y,
                mode="lines",
                line=dict(color=MUTED, width=0.9),
                opacity=0.60,
                hoverinfo="skip",
                showlegend=False,
            ),

            # One-dimensional projection distributions.
            go.Bar(
                x=centres,
                y=honest_hist,
                width=bin_width,
                marker=dict(color=C_HONEST),
                opacity=0.72,
                xaxis="x2",
                yaxis="y2",
                hovertemplate=(
                    "honest<br>"
                    "u·x ≈ %{x:.2f}<br>"
                    "share %{y:.1%}<extra></extra>"
                ),
                showlegend=False,
            ),
            go.Bar(
                x=centres,
                y=deceptive_hist,
                width=bin_width,
                marker=dict(color=C_DECEPTIVE),
                opacity=0.72,
                xaxis="x2",
                yaxis="y2",
                hovertemplate=(
                    "deceptive<br>"
                    "u·x ≈ %{x:.2f}<br>"
                    "share %{y:.1%}<extra></extra>"
                ),
                showlegend=False,
            ),

            # Current location on the AUC curve.
            go.Scatter(
                x=[angle],
                y=[auc],
                mode="markers",
                xaxis="x3",
                yaxis="y3",
                marker=dict(
                    size=13,
                    color=C_PROBE,
                    line=dict(color=PAPER, width=2),
                ),
                hovertemplate=f"{angle}° · AUC {auc:.3f}<extra></extra>",
                showlegend=False,
            ),

            # AUC readout as a trace so it updates correctly between frames.
            go.Scatter(
                x=[
                    score_range[0]
                    + 0.05 * (score_range[1] - score_range[0])
                ],
                y=[0.93 * hist_ymax],
                mode="text",
                xaxis="x2",
                yaxis="y2",
                text=[f"<b>AUC {auc:.3f}</b>"],
                textposition="middle right",
                textfont=dict(size=13, color=C_PROBE),
                hoverinfo="skip",
                showlegend=False,
            ),
        ]

    def circular_distance(a, b):
        return abs((a - b + 180) % 360 - 180)

    # Begin at the intuitive mean-difference direction rather than the fitted
    # direction.
    mean_angle = ang_of(w_diff)
    start = int(
        min(frame_deg, key=lambda a: circular_distance(a, mean_angle))
    )

    # Close the periodic curve at 360°.
    curve_angles = np.r_[angles, 360]
    curve_auc = np.r_[auc_by_angle, auc_by_angle[0]]

    static_traces = [
        go.Scatter(
            x=X[label == 0, 0],
            y=X[label == 0, 1],
            mode="markers",
            name="honest",
            marker=dict(
                size=5.5,
                color=C_HONEST,
                opacity=0.70,
                line=dict(width=0.5, color=PAPER),
            ),
            hovertemplate="honest<extra></extra>",
        ),
        go.Scatter(
            x=X[label == 1, 0],
            y=X[label == 1, 1],
            mode="markers",
            name="deceptive",
            marker=dict(
                size=5.5,
                color=C_DECEPTIVE,
                opacity=0.70,
                line=dict(width=0.5, color=PAPER),
            ),
            hovertemplate="deceptive<extra></extra>",
        ),
        go.Scatter(
            x=curve_angles,
            y=curve_auc,
            mode="lines",
            xaxis="x3",
            yaxis="y3",
            line=dict(color=C_PROBE, width=2),
            hovertemplate="%{x:.0f}° · AUC %{y:.3f}<extra></extra>",
            showlegend=False,
        ),

        # The intuitive direction.
        go.Scatter(
            x=[ang_of(w_diff)],
            y=[roc_auc_score(label, X @ w_diff)],
            mode="markers+text",
            xaxis="x3",
            yaxis="y3",
            marker=dict(size=9, color=MUTED, symbol="diamond"),
            text=["  mean difference"],
            textposition="middle right",
            textfont=dict(size=10, color=MUTED),
            hovertemplate="mean-difference direction<extra></extra>",
            showlegend=False,
        ),

        # The direction fitted by logistic regression.
        go.Scatter(
            x=[ang_of(w_fit)],
            y=[roc_auc_score(label, X @ w_fit)],
            mode="markers+text",
            xaxis="x3",
            yaxis="y3",
            marker=dict(size=10, color=C_PROBE, symbol="star"),
            text=["  fitted probe"],
            textposition="middle right",
            textfont=dict(size=10, color=C_PROBE),
            hovertemplate="fitted logistic direction<extra></extra>",
            showlegend=False,
        ),
    ]

    frames = [
        go.Frame(
            name=str(angle),
            data=dynamic_traces(angle),
            traces=list(range(7)),
        )
        for angle in frame_deg
    ]

    fig = go.Figure(
        data=dynamic_traces(start) + static_traces,
        frames=frames,
    )

    fig.update_layout(
        title=dict(
            text="A probe ranks examples along one oriented direction",
            x=0.005,
            y=0.975,
            yanchor="top",
            font=dict(size=15),
        ),
        height=560,
        margin=dict(l=50, r=34, t=118, b=100),
        dragmode=False,
        bargap=0.03,
        barmode="overlay",
        paper_bgcolor=PAPER,
        plot_bgcolor=PAPER,
        font=dict(
            family="Inter, -apple-system, Segoe UI, sans-serif",
            size=12,
            color=INK,
        ),
        legend=dict(
            orientation="h",
            y=1.055,
            x=0,
            bgcolor="rgba(0,0,0,0)",
            font=dict(size=11),
        ),

        # Activation cloud.
        xaxis=dict(
            domain=[0.0, 0.44],
            range=[-xy_radius, xy_radius],
            visible=False,
            anchor="y",
        ),
        yaxis=dict(
            domain=[0.0, 1.0],
            range=[-xy_radius, xy_radius],
            visible=False,
            anchor="x",
            scaleanchor="x",
            scaleratio=1,
        ),

        # Fixed-scale projection histograms.
        xaxis2=dict(
            domain=[0.55, 1.0],
            anchor="y2",
            range=list(score_range),
            gridcolor="#e8e8e4",
            title=dict(
                text="u · x — scalar projection",
                font=dict(size=11),
            ),
            tickfont=dict(size=10),
        ),
        yaxis2=dict(
            domain=[0.60, 1.0],
            anchor="x2",
            range=[0, hist_ymax],
            tickformat=".0%",
            gridcolor="#e8e8e4",
            title=dict(
                text="share within class",
                font=dict(size=11),
            ),
            tickfont=dict(size=10),
        ),

        # AUC as the direction rotates.
        xaxis3=dict(
            domain=[0.55, 1.0],
            anchor="y3",
            range=[0, 360],
            gridcolor="#e8e8e4",
            tickvals=[0, 90, 180, 270, 360],
            ticktext=["0°", "90°", "180°", "270°", "360°"],
            title=dict(
                text="orientation of u",
                font=dict(size=11),
            ),
            tickfont=dict(size=10),
        ),
        yaxis3=dict(
            domain=[0.0, 0.36],
            anchor="x3",
            range=[-0.03, 1.03],
            gridcolor="#e8e8e4",
            tickvals=[0, 0.5, 1],
            title=dict(text="AUC", font=dict(size=11)),
            tickfont=dict(size=10),
        ),

        shapes=[
            dict(
                type="line",
                xref="x3",
                yref="y3",
                x0=0,
                x1=360,
                y0=0.5,
                y1=0.5,
                line=dict(color=RULE, width=1.5, dash="dot"),
                layer="below",
            )
        ],
        annotations=[
            dict(
                x=0.55,
                y=1.07,
                xref="paper",
                yref="paper",
                text="Axes remain fixed while the direction rotates.",
                showarrow=False,
                xanchor="left",
                font=dict(size=11, color=MUTED),
            )
        ],
        sliders=[
            dict(
                active=int(np.where(frame_deg == start)[0][0]),
                currentvalue=dict(
                    prefix="orientation of u: ",
                    suffix="°",
                    font=dict(size=12, color=INK),
                ),
                pad=dict(t=44, b=6),
                x=0.0,
                len=0.44,
                steps=[
                    dict(
                        method="animate",
                        # Only label every 45° to avoid an unreadable slider.
                        label=str(angle) if angle % 45 == 0 else "",
                        args=[
                            [str(angle)],
                            dict(
                                mode="immediate",
                                frame=dict(duration=0, redraw=True),
                                transition=dict(duration=0),
                            ),
                        ],
                    )
                    for angle in frame_deg
                ],
            )
        ],
    )

    return fig


direction_figure().show(
    config={
        "displayModeBar": False,
        "responsive": True,
    },
    auto_play=False,
)

## 5. The real test: transfer and operational false positives

The instruction-pair probe has never seen model-generated deceptive behaviour. We now
apply it unchanged to the on-policy evaluation sets.
The probe turns each response into one number, higher meaning more deceptive. Two
questions follow, and they have different answers.

**1. Does it rank correctly? (cell 5A)** Do deceptive responses get higher scores than
honest ones from the same evaluation set? This is ROC-AUC = P(score of a random
deceptive response > score of a random honest one).

**2. Does it work as a detector? (cell 5B)** To decide about a single response you need
a threshold: score above it, the probe predicts *deceptive*; below it, *honest*.

We choose that threshold using the Alpaca control set  - a ordinary helpful requests, where
every response is honest. Any response the probe predicts deceptive there is simply a
mistake. We place the threshold so that exactly 1% of Alpaca is misclassified: the error
rate we are willing to accept. Then we apply that same threshold to the evaluation sets
and read two numbers:

- **Recall**  of the genuinely deceptive responses, the fraction the probe correctly
  predicts as deceptive. This is how much deception it detects.
- **False-positive rate on honest in-context responses**  of the *honest* responses
  inside the same deception-inviting scenario, the fraction the probe wrongly predicts
  as deceptive.

Compare that second number with the 1% we allowed on ordinary requests. It is
often higher: honest responses become much easier to mistake for lies once the
scenario itself invites deception and scenario responses are what a deployed monitor
would actually see.

Precision is not reported: it depends on how common deception is in the
traffic, and these evaluation sets are balanced by construction.


In [ ]:
# @title 5A. Transfer ROC-AUC with bootstrap intervals
def bootstrap_auc_ci(y, scores, n_boot=1000, alpha=0.05, seed=2026):
    y = np.asarray(y)
    scores = np.asarray(scores)
    indices = np.arange(len(y))
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(n_boot):
        sample = rng.choice(indices, size=len(indices), replace=True)
        if len(np.unique(y[sample])) == 2:
            boot.append(roc_auc_score(y[sample], scores[sample]))
    return tuple(np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


transfer_rows = []
fig = go.Figure()
for dataset in EVAL_SETS:
    if dataset not in D:
        continue
    y_eval = D[dataset]["labels"]
    s_eval = probe_scores(baseline_probe, dataset)
    auc = roc_auc_score(y_eval, s_eval)
    low, high = bootstrap_auc_ci(y_eval, s_eval)
    fpr, tpr, _ = roc_curve(y_eval, s_eval)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{dataset.replace('apollo_', '')} — {auc:.3f}",
        line=dict(width=2),
        hovertemplate="FPR %{x:.2f}<br>TPR %{y:.2f}<extra></extra>",
    ))
    transfer_rows.append(dict(
        dataset=dataset.replace("apollo_", ""), n=len(y_eval),
        deceptive_fraction=(y_eval == 1).mean(), auc=auc,
        ci_low=low, ci_high=high,
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="chance",
    line=dict(color=C_MUTED, width=1, dash="dot"), hoverinfo="skip",
))
fig.update_layout(
    title="One instruction-pair probe, transferred without retraining",
    xaxis_title="false-positive rate", yaxis_title="true-positive rate",
    height=430, autosize=True,
)
fig.show(config={"responsive": True})

display(pd.DataFrame(transfer_rows).round(3))


In [ ]:
# @title 5B. Threshold, recall, and false-positive rates
# Choose the evaluation dataset shown in the score-distribution plot.
MONITOR_DATASET = "apollo_roleplaying"  # @param ["apollo_sandbagging", "apollo_roleplaying", "apollo_ai_liar"]

# The threshold is set on the control set only: every Alpaca response is honest, so
# the 99th percentile is the score above which 1% of honest ordinary traffic is
# predicted deceptive -- the false-positive rate we have chosen to accept.
control_scores = probe_scores(
    baseline_probe, CONTROL, probabilities=False
)
operational_threshold = float(np.quantile(control_scores, 0.99))

y_monitor = D[MONITOR_DATASET]["labels"]
monitor_scores = probe_scores(
    baseline_probe, MONITOR_DATASET, probabilities=False
)

fig = go.Figure()
for values, name, colour in [
    (control_scores, "ordinary control (all honest)", C_MUTED),
    (monitor_scores[y_monitor == 0], "honest in-context", C_HONEST),
    (monitor_scores[y_monitor == 1], "deceptive", C_DECEPTIVE),
]:
    fig.add_trace(go.Histogram(
        x=values, name=name, marker=dict(color=colour), opacity=0.55,
        histnorm="probability", nbinsx=38,
        hovertemplate="probe score %{x:.2f}<br>share %{y:.2%}<extra></extra>",
    ))

fig.add_vline(
    x=operational_threshold, line=dict(color=C_INK, width=2, dash="dash"),
    annotation_text="threshold — 99th percentile of control (1% false positives)",
    annotation_position="top right",
)
fig.update_layout(
    title=(
        f"Threshold set on control, applied to "
        f"{MONITOR_DATASET.replace('apollo_', '')}"
    ),
    xaxis_title="probe score (higher = more deceptive) — right of the thresholdは predicted deceptive",
    yaxis_title="share within each group", barmode="overlay",
    height=430, autosize=True,
)
fig.show(config={"responsive": True})

monitor_rows = []
for dataset in EVAL_SETS:
    if dataset not in D:
        continue
    y_eval = D[dataset]["labels"]
    s_eval = probe_scores(baseline_probe, dataset, probabilities=False)
    monitor_rows.append(dict(
        dataset=dataset.replace("apollo_", ""),
        recall_deceptive=(s_eval[y_eval == 1] > operational_threshold).mean(),
        fpr_honest_in_context=(s_eval[y_eval == 0] > operational_threshold).mean(),
        roc_auc=roc_auc_score(y_eval, s_eval),
    ))

print(f"False-positive rate on ordinary control traffic: "
      f"{(control_scores > operational_threshold).mean():.1%}  (target 1%)")
display(pd.DataFrame(monitor_rows).round(3))

The control threshold is an operational choice, not a universal property of the
probe.  Notice also
that high AUC can coexist with an undesirable false-positive rate at the chosen
threshold.


## 6. Targeted probes: define the threat before seeing the answer

Natarajan et al. found that the training instruction explained much more performance
variation than layer or token aggregation. The seven targeted prompts here describe:

| Probe | Intended behavior |
|---|---|
| `overt_lie` | direct fabrication |
| `concealment` | hiding relevant information |
| `evasion` | avoiding a direct answer |
| `exaggeration` | overstating or amplifying a claim |
| `partial_truth` | selectively truthful but misleading information |
| `pressure_dishonesty` | deception under social or strategic pressure |
| `white_lie` | deception framed as protecting someone |

Choosing the best probe **after** seeing every test AUC is an oracle analysis. It
estimates a ceiling, not deployable performance.


In [ ]:
# @title 6A. Train one probe per instruction type
# ── Train one probe per instruction type ─────────────────────────────────────
PROBES = {
    name: fit_probe(name)
    for name in TRAIN_SETS
    if name in D
}

print(f"Trained {len(PROBES)} probes:")
for name in PROBES:
    print("  ", name)


### Exercise 3 — Commit to a targeted probe

Choose one evaluation dataset and one lie type **before running the next cell**.
Try to guess which type of probe would perfform best on which data set.


In [ ]:
TARGET_DATASET = "apollo_sandbagging"  # @param ["apollo_sandbagging", "apollo_roleplaying", "apollo_ai_liar"]
CHOSEN_LIE_TYPE = "exaggeration"  # @param ["overt_lie", "concealment", "evasion", "exaggeration", "partial_truth", "pressure_dishonesty", "white_lie"]

chosen_name = f"lietype_{CHOSEN_LIE_TYPE}"
if chosen_name not in PROBES:
    raise ValueError(f"No stored probe called {chosen_name}")

y_target = D[TARGET_DATASET]["labels"]
baseline_target_auc = roc_auc_score(
    y_target, probe_scores(PROBES["instructed_pairs"], TARGET_DATASET)
)
chosen_target_auc = roc_auc_score(
    y_target, probe_scores(PROBES[chosen_name], TARGET_DATASET)
)

print("Pre-committed target :", TARGET_DATASET)
print("Pre-committed probe  :", CHOSEN_LIE_TYPE)
print(f"Generic baseline AUC : {baseline_target_auc:.3f}")
print(f"Chosen probe AUC     : {chosen_target_auc:.3f}")
print(f"Difference           : {chosen_target_auc - baseline_target_auc:+.3f}")


In [ ]:
# @title 6B. Reveal the full probe-by-dataset heatmap
# ── Interactive probe-by-dataset comparison ──────────────────────────────────
AUC_MATRIX = pd.DataFrame(
    index=list(PROBES),
    columns=[dataset for dataset in EVAL_SETS if dataset in D],
    dtype=float,
)

for probe_name, probe in PROBES.items():
    for dataset in AUC_MATRIX.columns:
        AUC_MATRIX.loc[probe_name, dataset] = roc_auc_score(
            D[dataset]["labels"],
            probe_scores(probe, dataset),
        )

pretty_rows = [
    name.replace("lietype_", "").replace("instructed_pairs", "BASELINE")
    for name in AUC_MATRIX.index
]
pretty_columns = [
    name.replace("apollo_", "")
    for name in AUC_MATRIX.columns
]

fig = go.Figure(go.Heatmap(
    z=AUC_MATRIX.values,
    x=pretty_columns,
    y=pretty_rows,
    colorscale=DIV_AUC,
    zmid=0.5,
    zmin=0.3,
    zmax=1.0,
    xgap=2,
    ygap=2,
    colorbar=dict(title="AUC", thickness=14, len=0.75),
    hovertemplate="training probe %{y}<br>evaluation set %{x}<br>"
                  "AUC %{z:.3f}<extra></extra>",
    text=[[f"{value:.2f}" for value in row] for row in AUC_MATRIX.values],
    texttemplate="%{text}",
    textfont=dict(size=12),
))
fig.update_layout(
    title="A probe trained for one lie type does not transfer equally to every setting",
    xaxis_title="evaluation dataset",
    yaxis_title="training instruction",
    height=470, autosize=True, margin=dict(l=155, r=35, t=80, b=70),
)
fig.show(config={"responsive": True})

AUC_MATRIX.round(3)


In [ ]:
# @title 6C. Generic baseline versus post-hoc oracle ceiling
lie_type_rows = [name for name in AUC_MATRIX.index if name.startswith("lietype_")]
oracle_rows = []
for dataset in AUC_MATRIX.columns:
    baseline = AUC_MATRIX.loc["instructed_pairs", dataset]
    best_name = AUC_MATRIX.loc[lie_type_rows, dataset].idxmax()
    best_value = AUC_MATRIX.loc[best_name, dataset]
    oracle_rows.append(dict(
        dataset=dataset.replace("apollo_", ""),
        generic_baseline=baseline,
        posthoc_oracle=best_value,
        oracle_probe=best_name.replace("lietype_", ""),
        oracle_gain=best_value - baseline,
    ))

oracle = pd.DataFrame(oracle_rows)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=oracle["dataset"], y=oracle["generic_baseline"],
    name="generic baseline", marker=dict(color=C_HONEST),
    text=[f"{v:.2f}" for v in oracle["generic_baseline"]], textposition="outside",
))
fig.add_trace(go.Bar(
    x=oracle["dataset"], y=oracle["posthoc_oracle"],
    name="post-hoc oracle (optimistic ceiling)", marker=dict(color=C_DECEPTIVE),
    text=[f"{v:.2f}<br>{p}" for v, p in zip(oracle["posthoc_oracle"], oracle["oracle_probe"])],
    textposition="outside",
))
fig.add_hline(y=0.5, line=dict(color=C_MUTED, width=1, dash="dot"))
fig.update_layout(
    title="Post-hoc oracle gain: a descriptive ceiling, not a deployment estimate",
    yaxis_title="AUC", yaxis_range=[0.3, 1.08], barmode="group",
    height=430, autosize=True,
)
fig.show(config={"responsive": True})
display(oracle.round(3))


## 7. Takeaways

1. **Linear probes reveal decodability.** They show that label-related information can
   be read with a linear function; they do not establish a uniquely identified or
   causally used representation.
2. **Within-distribution success is only the first test.** Matched groups must stay in
   the same fold, and label shuffling should return to chance.
3. **Transfer and controls change the conclusion.** AUC measures ranking; an operational
   monitor also needs calibrated thresholds and acceptable false positives.
4. **Deception is heterogeneous.** Prompt-targeted probes should be chosen from an
   explicit threat model and validated on held-out matched scenarios.
5. **The labels and token window matter.** Rules, LLM judges, prompt wording, visible
   reasoning, and pooling choices can all create shortcuts.

---

# Appendix

Some additional details that were skipped in the main tutorial.


## A. Full five-dataset gallery


In [ ]:
# @title A1. Print one honest and deceptive example per dataset
# ── The data gallery: one honest and one deceptive example per dataset ───────
import textwrap

DESCRIPTION = {
    "instructed_pairs": (
        "TRAIN — token-forced instruction pairs",
        "Same factual sentence, opposite system prompts. The response is INSERTED by us, not "
        "generated. Label = which instruction was used."),
    "apollo_sandbagging": (
        "EVAL — strategic underperformance",
        "Told it is being evaluated on dangerous capabilities; reasons in <step> tags and may "
        "choose to answer worse than it can. Label = RULE-BASED parse of those tags, no judge."),
    "apollo_roleplaying": (
        "EVAL — a role with an incentive to shade the truth",
        "A scenario puts the model in a role where honesty is costly. Label = LLM judge on "
        "Apollo's 1-7 honesty rubric; scores of 3, 4, 5 are dropped as ambiguous."),
    "apollo_ai_liar": (
        "EVAL — goal-directed lying",
        "Each scenario has an honest and a deceptive instruction variant. Label = LLM judge, "
        "Apollo's categorical rubric."),
    "alpaca_control": (
        "CONTROL — ordinary instruction following",
        "No deceptive condition is introduced. These are ordinary, non-deception-related "
        "responses used as a control; they are not guaranteed to be factually correct."),
}

def show(text, width=104, indent="    "):
    for para in str(text).split("\n"):
        for line in textwrap.wrap(para, width=width) or [""]:
            print(indent + line)

def gallery(name, max_chars=520):
    records = TEXT.get(name, [])
    title, blurb = DESCRIPTION.get(name, (name, ""))
    print("=" * 108)
    print(f"{name}   —   {title}")
    print("-" * 108)
    show(blurb, indent="  ")
    if not records:
        print("  (no .jsonl found for this dataset)\n")
        return
    for want in (0, 1):
        pick = next((r for r in records if r.get("label") == want), None)
        if pick is None:
            print(f"\n  [no examples with label={want} in this dataset]")
            continue
        print(f"\n  ── label = {want} ({LABEL_NAME[want]}) " + "─" * 74)
        for field in ("system", "user", "response"):
            value = str(pick.get(field, ""))
            clipped = value[:max_chars] + (" …[truncated]" if len(value) > max_chars else "")
            print(f"  {field}:")
            show(clipped)
        if pick.get("judge_model"):
            print(f"  labeled by: {pick['judge_model']}")
    print()

for name in ["instructed_pairs", "apollo_sandbagging",
             "apollo_roleplaying", "apollo_ai_liar", "alpaca_control"]:
    gallery(name)


## B. Layer-norm and layer-sweep analysis

Layer is a hyperparameter. The within-training curves below show probe performance on different layers.

In [ ]:
# @title B1. Grouped-CV layer explorer
# ── Layer explorer: does the layer choice matter? ────────────────────────────
# Buttons, not a slider. A slider implies a continuous quantity you can glide
# through; these are 7 discrete taps into the network. Match the control to the
# data.
#
# What is plotted: the probe's own decision function -- the single axis it scores
# along -- as one histogram per class, with the layer-by-layer AUC beside it. The
# tempting alternative is a PCA scatter of the activations, but PCA is
# unsupervised: it draws the directions of largest variance, which need not have
# anything to do with the label, so it shows overlapping blobs even where the
# probe scores 0.99. A picture that contradicts the number next to it teaches the
# wrong thing. Plot the axis the classifier actually uses.
from sklearn.model_selection import cross_val_predict

data = D["instructed_pairs"]
y = data["labels"]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

layer_scores, layer_aucs = [], []
for layer_index, layer in enumerate(LAYERS):
    X = data["acts"][:, layer_index, :]
    # `decision_function` is the signed distance along the probe's own direction.
    # `cross_val_predict` scores every example with a probe that never saw it, so
    # the histogram and the AUC beside it describe the same held-out predictions
    # -- no train-set optimism sneaking into the picture.
    s = cross_val_predict(
        make_probe(), X, y, groups=PAIR_GROUPS, cv=cv, method="decision_function"
    )
    # Standardise the scores before plotting. Each layer's raw decision function
    # has its own scale, so without this the x axis silently rescales between
    # layers and two panels that look equally separated are not. AUC is invariant
    # to any increasing transform, so this changes the picture, not the number.
    layer_scores.append((s - s.mean()) / s.std())
    layer_aucs.append(roc_auc_score(y, s))

# No subplot titles: the top margin already carries a title, a caption, the
# button row and the legend, and a fifth text element in that stack is what makes
# plotly figures look crowded. The axis titles say what each panel shows.
fig = make_subplots(rows=1, cols=2, column_widths=[0.56, 0.44], horizontal_spacing=0.11)

# Three traces per layer (two histograms + the marker that highlights that layer
# on the right-hand curve); the AUC curve itself is added last and stays visible.
for i, layer in enumerate(LAYERS):
    for k, colour in [(0, C_HONEST), (1, C_DECEPTIVE)]:
        fig.add_trace(go.Histogram(
            x=layer_scores[i][y == k], nbinsx=34, marker=dict(color=colour),
            opacity=0.62, name=LABEL_NAME[k], legendgroup=LABEL_NAME[k],
            visible=(i == LI), showlegend=(i == LI),
            hovertemplate=f"layer {layer}<br>score %{{x:.2f}}<extra></extra>"),
            row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[layer], y=[layer_aucs[i]], mode="markers",
        marker=dict(size=15, color="rgba(0,0,0,0)", line=dict(width=2.5, color=C_INK)),
        visible=(i == LI), showlegend=False, hoverinfo="skip"), row=1, col=2)

fig.add_trace(go.Scatter(
    x=LAYERS, y=layer_aucs, mode="lines+markers",
    line=dict(width=2, color=C_MUTED), marker=dict(size=8, color=C_MUTED),
    showlegend=False, hovertemplate="layer %{x}<br>AUC %{y:.3f}<extra></extra>"),
    row=1, col=2)

n_toggled = 3 * len(LAYERS)


def visibility(active):
    """Three traces per layer, then the always-on AUC curve."""
    return [i // 3 == active if i < n_toggled else True
            for i in range(len(fig.data))]


def legend_flags(active):
    """Only the two histograms belong in the legend -- never the highlight ring
    or the AUC curve. Reusing `visibility` here puts a stray 'trace 2' entry in
    the legend the moment you click a button."""
    return [i // 3 == active and i % 3 < 2 if i < n_toggled else False
            for i in range(len(fig.data))]


buttons = [
    dict(label=f"{layer}★" if layer == PRIMARY_LAYER else str(layer),
         method="update",
         # "title.text", not "title": passing a plain string replaces the whole
         # title object and silently discards its x/y positioning, so the title
         # jumps back under the button row on the first click.
         args=[{"visible": visibility(i), "showlegend": legend_flags(i)},
               {"title.text": f"Layer {layer} — held-out AUC {layer_aucs[i]:.3f}"}])
    for i, layer in enumerate(LAYERS)
]

# Vertical stack in the top margin, top to bottom: title, caption, buttons,
# legend. Each needs its own band or they overlap -- and because "paper" y=1 is
# the top of the PLOT area, every one of these bands has to be > 1.
fig.update_layout(
    title=dict(text=f"Layer {PRIMARY_LAYER} — held-out AUC {layer_aucs[LI]:.3f}",
               x=0, xanchor="left", y=0.975, yref="container"),
    barmode="overlay", height=500, margin=dict(t=185),
    legend=dict(orientation="h", y=1.07, x=0),
    updatemenus=[dict(
        type="buttons", direction="right", active=LI,
        x=0, xanchor="left", y=1.34, yanchor="top",
        pad=dict(r=4, t=4), showactive=True,
        bgcolor="#fcfcfb", bordercolor="#d8d7d2", borderwidth=1,
        font=dict(size=12), buttons=buttons)],
    annotations=[dict(
        x=0, y=1.42, xref="paper", yref="paper", showarrow=False, xanchor="left",
        font=dict(size=12, color=C_MUTED),
        text="layer: &nbsp;(★ = the layer used everywhere else in this notebook)")])
lim = max(abs(s).max() for s in layer_scores) * 1.05
fig.update_xaxes(title_text="probe score, in standard deviations "
                            "(signed distance along the probe direction)",
                 range=[-lim, lim], row=1, col=1)
fig.update_yaxes(title_text="examples", row=1, col=1)
fig.update_xaxes(title_text="layer", row=1, col=2)
fig.update_yaxes(title_text="AUC", range=[0.45, 1.02], row=1, col=2)
fig.show()

# Say out loud what the curve shows, computed rather than asserted -- if you
# re-run this on your own data the sentence has to stay true.
deep = [(layer, auc) for layer, auc in zip(LAYERS, layer_aucs) if layer >= 8]
spread = max(a for _, a in deep) - min(a for _, a in deep)
best_layer, best_auc = max(deep, key=lambda t: t[1])

print("Held-out AUC by layer:")
for layer, auc in zip(LAYERS, layer_aucs):
    bar = "█" * int(round((auc - 0.5) * 60)) if auc > 0.5 else ""
    star = "  <- the layer we use" if layer == PRIMARY_LAYER else ""
    print(f"  layer {layer:2d}  {auc:.3f}  {bar}{star}")



## References

- Natarajan, V., Jain, D., Arora, S., Golechha, S., & Bloom, J. (2026).
  [*One Probe Won't Catch Them All: Towards Targeted Deception Detection*](https://arxiv.org/abs/2602.01425).
  ICML 2026.
- Goldowsky-Dill, N., Chughtai, B., Heimersheim, S., & Hobbhahn, M. (2025).
  [*Detecting Strategic Deception with Linear Probes*](https://arxiv.org/abs/2502.03407).
  ICML 2025.

### Acknowledgements


*The preparation of the notebook is supported by Coefficinet Giving.*